In [ ]:
import numpy as np
from mlx_audio.audio_io import write as write_audio
from mlx_audio.stt.utils import load_audio

from mlx_vlm import load

model, processor = load("mlx-community/NemotronLabs-VoiceChat-11B-8bit")
voicechat = model.create_session(processor)
stream = voicechat.create_streaming_session(
    system_prompt="Be concise and answer in one sentence.",
    seed=0,
)

input_audio = load_audio("input.wav", sr=16_000).squeeze()
audio_chunks = []


def handle(events):
    for event in events:
        if event.kind == "assistant_text_delta":
            print(event.delta or "", end="", flush=True)
        elif event.kind == "user_transcript_delta":
            print(f"\n[user] {event.text}")
        elif event.kind == "function_delta":
            print(f"\n[function] {event.delta}")
        elif event.kind == "audio":
            audio_chunks.append(np.asarray(event.samples, dtype=np.float32))


for offset in range(0, input_audio.shape[0], 1_280):
    handle(
        stream.push_audio(
            input_audio[offset : offset + 1_280],
            sample_rate=16_000,
        )
    )
handle(stream.flush(pad_partial=True))

response_audio = (
    np.concatenate(audio_chunks) if audio_chunks else np.zeros(0, dtype=np.float32)
)
write_audio("response.wav", response_audio, 22_050)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

ERROR:root:Model type nemotron_voicechat not supported. Error: No module named 'mlx_vlm.speculative.drafters.nemotron_voicechat'


ValueError: Model type nemotron_voicechat not supported. Error: No module named 'mlx_vlm.speculative.drafters.nemotron_voicechat'